In [1]:
# język - duński

# 1 prawo zipfa
# 2 rdzeń języka - jeśli 2 wyrazy wystepują obok siebie to łączymy wyrazy w grafie
# rdzeniem jest słowo/słowa które łączą się z wieloma wyrazami
# 3 ile słów należy znać, żeby zrozumieć 90% wyrazów napisanych w tekście (ponoć nie dużo)

# zadanie* (specjalne) - zidentyfikuj znaczenia 50 rzeczowników, w wielu językach te same rzeczowniki są uznawane za najważniejsze
# np. słońce, gwiazda, drzewo - takie słowa powinny znajdować się na podobnym miejscu w liście

In [2]:
# 1 zebrać teksty

In [3]:
import numpy
import mwparserfromhell
import re
from collections import defaultdict
import json
import xml.etree.ElementTree as ET

path = "dawiki-latest-pages-articles-multistream.xml"
ns = "{http://www.mediawiki.org/xml/export-0.11/}"
iterations = 0

words = defaultdict(int)
core = defaultdict(set)

for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag == ns + "page":
        # tytuł
        title = elem.find(ns + "title").text

        # surowa zawartość artykułu
        text_elem = elem.find(f"{ns}revision/{ns}text")
        if text_elem is None or not text_elem.text:
            elem.clear()
            continue

        wikicode = text_elem.text

        # parsowanie MediaWiki i usuwanie znaczników
        parsed = mwparserfromhell.parse(wikicode)
        clean_text = parsed.strip_code()  # usuwa linki, nagłówki, obrazy itp.

        # można też usunąć puste linie
        clean_text = "\n".join([line.strip() for line in clean_text.splitlines() if line.strip()])

        # print(f"=== {title} ===")
        # print(wikicode)

        clean_text = clean_text.lower()

        clean_text = re.sub(r"[^a-zæøå \n.]", "", clean_text)

        sentences = re.split(r"[.\n]+", clean_text)

        for i in range(len(sentences)):
            sentences[i] = sentences[i].strip()

        for sentence in sentences:
            prev_word = ''
            for word in sentence.split():
                words[word] += 1
                if prev_word != '':           
                    core[word].add(prev_word)
                    core[prev_word].add(word)
                prev_word = word
                
                
                    
        # print(sentences)
        iterations += 1

        elem.clear()
        
        if iterations % 10000 == 0 and iterations != 0:
            print(iterations)

        # if iterations == 4000:
        #     break

core = dict(core)
for word in core.keys():
    core[word] = len(core[word])

topwords_10000 = dict(sorted(words.items(), key=lambda x: x[1], reverse=True)[:10000])
topcore_10000 = dict(sorted(core.items(), key=lambda x: x[1], reverse=True)[:10000])
total_words = sum(words.values())

with open("topwords_10000.json", "w", encoding="utf-8") as f:
    json.dump(topwords_10000, f, ensure_ascii=False, indent=2)

with open("topcore_10000.json", "w", encoding="utf-8") as f:
    json.dump(topcore_10000, f, ensure_ascii=False, indent=2)

total_chance = 0
words_needed = 0
sorted_words = sorted(words.items(), key=lambda x: x[1], reverse=True)
for word, count in sorted_words:
    chance = count/total_words
    if total_chance < 0.9:
        total_chance += chance
        words_needed += 1
    else:
        break

words_json = defaultdict(list)

for index, word in enumerate(topwords_10000.keys()):
    chance = words[word]/total_words
    # f.write(f"{word}, {chance:.5f}%, {(index+1)*chance:.5f}\n")
    words_json[index] = [word, f"{chance:.5f}", f"{(index + 1) * chance:.5f}"]

with open("zipf.json", "w", encoding="utf-8") as f:
    json.dump(words_json, f, ensure_ascii=False, indent=2)

with open("info.txt", "w") as f:
    f.write(str(len(words)) + " differend words in dutch wikipedia\n")
    f.write(str(total_words) + " total words in dutch wikipedia\n")
    f.write(str(words_needed) + " words needed to understand " + str(total_chance) + " of language\n")



        

10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
